# v8 Unified Growth-Model Notebook

Date authored: 2026-06-01
Method reference: `01_growth_model/README.md`

## Scope

This is the active v8 growth-model workflow on the source workbook
`01_growth_model/input/2026_global_raw_data_20260601.xlsx`. The equations,
filters, and scenario construction are fixed for the paper-facing run; the
country pool and workbook anchors are defined by the v8 input README.

The active pool has ten members (UK, US, EU, China, Middle East, Australia,
Canada, Indonesia, Thailand, Brazil). `EU` denotes the workbook's
`EU without UK` sheet. Canada now carries a published policy target. All 8
scenarios (S1-S8) are covered with both Logistic and Gompertz S-curves.

The legacy `01_growth_model/code/run_logistic_mcmc_from_scenarios.py` covers
only S1-S4 (technical scenarios) and is superseded by this notebook for any
paper-facing analysis.

## What this notebook produces

All outputs go to `01_growth_model/output/v8_2026-06-01/`:

- `samples_{scenario}.csv`: accepted MC samples per scenario
- `acceptance.csv`: attempted/accepted counts per (s, i, m)
- `bands.csv`: pointwise percentile uncertainty envelopes
- `scalars_central.csv`: deterministic central scalars
- `timeseries_central.csv`: deterministic central annual series
- `sobol_indices.csv`: Saltelli Sobol on unconstrained prior
- `anova_conditional.csv`: ANOVA partial R² on accepted set
- `global_bands.csv` / `global_central.csv`: bootstrap global aggregation
- `q25_q75_sample_paths.csv`: quartile-representative actual sample paths
- `feasibility.csv`: infeasibility and fallback logs
- `figures/`: main-text figures plus supplementary dossiers/dashboards

## Non-blocking risks acknowledged

1. `mcmc_input_ipcc_low.csv` and `mcmc_input_ipcc_high.csv` are the same
   content by design. Section 1 below includes a runtime assertion that
   they remain identical.
2. The 2050 anchors, storage resources, and policy/IPCC targets are taken
   from the 2026-06-01 workbook; see the input-refresh README for the
   per-country verification notes.


In [1]:
# ===========================================================
# Section 0: Imports, paths, constants
# ===========================================================
import hashlib
import math
import warnings
from itertools import product
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy.optimize import brentq

warnings.filterwarnings('ignore', category=RuntimeWarning)

# ---- Paths ----
BASE_DIR  = Path('/Users/xg320/Desktop/IC-Sam-works/paper/Iman-2026')
INPUT_DIR = BASE_DIR / '01_growth_model' / 'output' / 'mcmc_inputs'
OUT_DIR   = BASE_DIR / '01_growth_model' / 'output' / 'v8_2026-06-01'
FIG_DIR   = OUT_DIR / 'figures'
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

# ---- Scenario config (per r4 method §2) ----
SCENARIO_FILES = {
    'reference':  INPUT_DIR / 'mcmc_input_reference.csv',
    'minimum':    INPUT_DIR / 'mcmc_input_minimum.csv',
    'maximum':    INPUT_DIR / 'mcmc_input_maximum.csv',
    'growth10':   INPUT_DIR / 'mcmc_input_growth10.csv',
    'us1gt':      INPUT_DIR / 'mcmc_input_us1gt.csv',
    'policy':     INPUT_DIR / 'mcmc_input_policy.csv',
    'ipcc_low':   INPUT_DIR / 'mcmc_input_ipcc_low.csv',
    'ipcc_high':  INPUT_DIR / 'mcmc_input_ipcc_high.csv',
}

# Which column of the anchored CSV holds the point target.
# For us1gt/policy the two columns are equal; for ipcc_low we read p2050_lo
# and for ipcc_high we read p2050_hi (dual-file / single-source convention).
SCENARIO_TARGET_COLUMN = {
    'us1gt':     'p2050_lo',   # == p2050_hi
    'policy':    'p2050_lo',   # == p2050_hi
    'ipcc_low':  'p2050_lo',
    'ipcc_high': 'p2050_hi',
}
TECHNICAL_SCENARIOS = ['reference', 'minimum', 'maximum', 'growth10']
ANCHORED_SCENARIOS  = ['us1gt', 'policy', 'ipcc_low', 'ipcc_high']

# ---- MC + filter constants (per r4 §4) ----
T_BASE       = 2030
T_END        = 2179
TP_MIN       = 2050
TP_MAX       = 2100
G_MIN        = 0.01       # Zhang lower bound on growth rate
N_SAMPLES    = 10_000     # Zhang MC count (D-unchanged)
K_SCREEN     = 200        # D3: robustness-screening sample count
B_BOOTSTRAP  = 1_000      # D5: bootstrap iterations
N_SOBOL_BASE = 4_096      # Saltelli base samples
BRENTQ_XTOL  = 1e-10
RESIDUAL_TOL = 1e-8

YEARS = np.arange(T_BASE, T_END + 1)
N_YEARS = len(YEARS)
IDX_2050 = int(2050 - T_BASE)
IDX_2100 = int(2100 - T_BASE)

PHI   = (3.0 + math.sqrt(5.0)) / 2.0   # Gompertz golden ratio
SQRT3 = math.sqrt(3.0)

print(f'Setup ready.')
print(f'  G_MIN = {G_MIN}, peak window = ({TP_MIN}, {TP_MAX})')
print(f'  N_SAMPLES = {N_SAMPLES:,}, K_SCREEN = {K_SCREEN}, B = {B_BOOTSTRAP:,}')
print(f'  Output: {OUT_DIR}')


Setup ready.
  G_MIN = 0.01, peak window = (2050, 2100)
  N_SAMPLES = 10,000, K_SCREEN = 200, B = 1,000
  Output: /Users/xg320/Desktop/IC-Sam-works/paper/Iman-2026/01_growth_model/output/v8_2026-06-01


In [2]:
# ===========================================================
# Section 0b: Model math
# ===========================================================

def logistic_cum(t, C, r, t0, S0):
    """True Logistic cumulative state at year t (model's integral, not a Riemann sum)."""
    k = (C - S0) / S0
    return C / (1.0 + k * np.exp(-r * (t - t0)))


def logistic_rate(t, C, r, t0, S0):
    """Instantaneous Logistic injection rate at year t."""
    S = logistic_cum(t, C, r, t0, S0)
    return r * S * (1.0 - S / C)


def gompertz_cum(tau, C, b, r):
    """True Gompertz cumulative state at elapsed time tau = t - t0."""
    return C * np.exp(-b * np.exp(-r * tau))


def logistic_inv(C, S0, t0, g):
    """Inverse calibration: given (C, S0, t0, g_CAGR), return (r, tn, tp, S_Tn)."""
    k    = (C - S0) / S0
    S_Tn = C / (3.0 + SQRT3)
    A    = math.log(k) - math.log(2.0 + SQRT3)
    r    = A * math.log(1.0 + g) / math.log(S_Tn / S0)
    tn   = t0 + A / r
    tp   = t0 + math.log(k) / r
    return r, tn, tp, S_Tn


def gompertz_rate_fn(tau, C, b, r):
    """Instantaneous Gompertz injection rate at elapsed time tau = t - t0."""
    s = b * np.exp(-r * tau)
    return C * r * s * np.exp(-s)


def gompertz_inv(C, S0, t0, g):
    """Inverse calibration: given (C, S0, t0, g_CAGR), return (r, b, tn, tp, P_Tn)."""
    b    = -math.log(S0 / C)
    P_Tn = C * math.exp(-PHI)
    B    = math.log(b / PHI)
    r    = B * math.log(1.0 + g) / math.log(P_Tn / S0)
    tn   = t0 + B / r
    tp   = t0 + math.log(b) / r
    return r, b, tn, tp, P_Tn


def C_lower_bound(model, S0):
    """Minimum C for the inverse calibration to be well-posed."""
    if model == 'Logistic':
        return S0 * (3.0 + SQRT3) * 1.01
    return S0 * math.exp(PHI) * 1.01


def curve_for_gC(model, C, S0, t0, g):
    """Forward trajectory on YEARS. Returns (r, tn, tp, rates, cum_state).

    `cum_state[i]` is the model's **exact** cumulative storage at
    `YEARS[i]` (not a discrete sum of rates).
    """
    if model == 'Logistic':
        r, tn, tp, _ = logistic_inv(C, S0, t0, g)
        rates = logistic_rate(YEARS, C, r, t0, S0)
        cum_state = logistic_cum(YEARS, C, r, t0, S0)
    else:
        r, b, tn, tp, _ = gompertz_inv(C, S0, t0, g)
        tau = YEARS - t0
        rates = gompertz_rate_fn(tau, C, b, r)
        cum_state = gompertz_cum(tau, C, b, r)
    return r, tn, tp, rates, cum_state


def parse_growth_pct(x):
    """Parse '20' or '20%' or 0.20 to fraction 0.20."""
    if isinstance(x, (int, float, np.integer, np.floating)):
        v = float(x)
    else:
        v = float(str(x).strip().rstrip('%'))
    return v / 100.0 if v > 1 else v


def stable_seed(*parts):
    """Deterministic 32-bit seed from hashed tag."""
    tag = '|'.join(str(p) for p in parts).encode('utf-8')
    return int.from_bytes(hashlib.sha256(tag).digest()[:4], 'big')


print('Model math ready.')


Model math ready.


## Section 1: Input loading and scenario classification

Per r4 §4 Step 0:

- Load all 8 `mcmc_input_{scenario}.csv`.
- Consistency assertion: `mcmc_input_ipcc_low.csv` and
  `mcmc_input_ipcc_high.csv` must have identical content (dual-file /
  single-source convention).
- Classify each `(scenario, country)` as Type A (no anchor) or
  Type B-fixed (workbook point target via the scenario-specific
  column).


In [3]:
# Load the 8 CSVs and assert the ipcc_low / ipcc_high consistency.

scenario_csvs = {}
for scenario, path in SCENARIO_FILES.items():
    df = pd.read_csv(path)
    df['Country'] = df['Country'].astype(str).str.strip()
    scenario_csvs[scenario] = df
    print(f'  {scenario:10s}  {len(df)} rows, {list(df.columns)[:4]}...')

# --- Consistency assertion: ipcc_low vs ipcc_high (non-blocking risk #1) ---
try:
    pd.testing.assert_frame_equal(
        scenario_csvs['ipcc_low'], scenario_csvs['ipcc_high'],
        check_exact=True,
    )
    print('\n[assert] ipcc_low and ipcc_high CSVs are byte-identical ✓')
except AssertionError as exc:
    raise AssertionError(
        'ipcc_low and ipcc_high CSVs differ! They must be identical content '
        'per the dual-file / single-source convention (see '
        'mcmc_inputs/README.md §Maintenance ownership). Re-sync the two files '
        'before continuing.'
    ) from exc


def classify_row(scenario, row):
    """Return ('A' | 'B_fixed', target_fixed_or_None)."""
    if scenario in TECHNICAL_SCENARIOS:
        return 'A', None
    col = SCENARIO_TARGET_COLUMN[scenario]
    val = row.get(col)
    if val is None or pd.isna(val):
        return 'A', None
    return 'B_fixed', float(val)


# Build (scenario, country) classification table
rows = []
for scenario, df in scenario_csvs.items():
    for _, r in df.iterrows():
        t, tgt = classify_row(scenario, r)
        rows.append({
            'Scenario': scenario, 'Country': r['Country'],
            'C_scenario': float(r['Storage capacity (Gt)']),
            'S_0': float(r['2030 Cumulative Storage (Gt)']),
            'rate_2030_Mt_yr': float(r['2030 Storage Rate (Mt/Year)']),
            'g_cap': parse_growth_pct(r['Growth_to_Tn_%']),
            'Type': t, 'target_fixed': tgt,
        })

scenario_country_table = pd.DataFrame(rows)
n_A       = (scenario_country_table['Type'] == 'A').sum()
n_Bfixed  = (scenario_country_table['Type'] == 'B_fixed').sum()
print(f'\nClassification: {n_A} Type A + {n_Bfixed} Type B-fixed = '
      f'{len(scenario_country_table)} total')

# Per-scenario Type counts (r4 Table §2)
print('\nPer-scenario Type counts:')
pivot = (scenario_country_table
         .groupby(['Scenario', 'Type']).size()
         .unstack(fill_value=0))
print(pivot.to_string())


  reference   10 rows, ['Country', 'Storage capacity (Gt)', '2030 Cumulative Storage (Gt)', '2030 Cumulative Storage (Mt)']...
  minimum     10 rows, ['Country', 'Storage capacity (Gt)', '2030 Cumulative Storage (Gt)', '2030 Cumulative Storage (Mt)']...
  maximum     10 rows, ['Country', 'Storage capacity (Gt)', '2030 Cumulative Storage (Gt)', '2030 Cumulative Storage (Mt)']...
  growth10    10 rows, ['Country', 'Storage capacity (Gt)', '2030 Cumulative Storage (Gt)', '2030 Cumulative Storage (Mt)']...
  us1gt       10 rows, ['Country', 'Storage capacity (Gt)', '2030 Cumulative Storage (Gt)', '2030 Cumulative Storage (Mt)']...
  policy      10 rows, ['Country', 'Storage capacity (Gt)', '2030 Cumulative Storage (Gt)', '2030 Cumulative Storage (Mt)']...
  ipcc_low    10 rows, ['Country', 'Storage capacity (Gt)', '2030 Cumulative Storage (Gt)', '2030 Cumulative Storage (Mt)']...
  ipcc_high   10 rows, ['Country', 'Storage capacity (Gt)', '2030 Cumulative Storage (Gt)', '2030 Cumulative St

## Section 2: Paired Monte Carlo sampling

Per r4 §4 Step 1. Uses the same `(g_j, target_j)` sample array across
Logistic and Gompertz so every MC sample yields a pair of trajectories on
the same geological footing.

- Type A: `g ~ U(G_MIN, g_cap)`, `target ~ U(0.001, target_hi_common)`
  where `target_hi_common = min(target_hi_L, target_hi_G)`.
- Type B-fixed: `g ~ U(G_MIN, g_cap)`, `target ≡ target_fixed`
  (constant, effectively 1-DOF MC).

Each `(model m, sample j)` applies brentq on
`rate_m(2050 | C, S0, g_j) = target_j` to find `C_sampled_{j,m}`. Accept
if converged, `C_sampled ≤ C_scenario`, and `2050 < tp < 2100`.


In [4]:
def target_hi_common_for(S0, C_scenario, g_cap):
    """Compute model-agnostic upper bound for Type A target prior."""
    # rate at 2050 when g = g_cap, C = C_scenario (both models)
    try:
        _, _, _, rates_L, _ = curve_for_gC('Logistic', C_scenario, S0, T_BASE, g_cap)
        target_hi_L = float(rates_L[IDX_2050])
    except (ValueError, ZeroDivisionError):
        target_hi_L = np.inf
    try:
        _, _, _, rates_G, _ = curve_for_gC('Gompertz', C_scenario, S0, T_BASE, g_cap)
        target_hi_G = float(rates_G[IDX_2050])
    except (ValueError, ZeroDivisionError):
        target_hi_G = np.inf
    return float(min(target_hi_L, target_hi_G))


def solve_C_for_target(model, S0, g, target, C_scenario):
    """brentq for C such that rate_m(2050 | C, S0, g) = target.

    Returns (C, r, tn, tp, rates) or None if infeasible.
    """
    C_lo = C_lower_bound(model, S0)
    if C_lo >= C_scenario:
        return None

    def residual(C):
        try:
            _, _, _, rates, _ = curve_for_gC(model, C, S0, T_BASE, g)
        except (ValueError, ZeroDivisionError):
            return float('nan')
        return float(rates[IDX_2050]) - target

    f_lo = residual(C_lo)
    f_hi = residual(C_scenario)
    if not (np.isfinite(f_lo) and np.isfinite(f_hi)):
        return None
    if f_lo * f_hi > 0:
        return None

    try:
        C_sol = brentq(residual, C_lo, C_scenario, xtol=BRENTQ_XTOL,
                       rtol=1e-12, maxiter=200)
    except Exception:
        return None

    try:
        r, tn, tp, rates, cum = curve_for_gC(model, C_sol, S0, T_BASE, g)
    except (ValueError, ZeroDivisionError):
        return None

    if not (np.isfinite(r) and np.isfinite(tn) and np.isfinite(tp)):
        return None
    if abs(float(rates[IDX_2050]) - target) > RESIDUAL_TOL:
        return None
    return C_sol, r, tn, tp, rates, cum


def mc_pool(scenario, country, C_scenario, S0, g_cap, row_type, target_fixed):
    """Generate paired MC pools for Logistic and Gompertz.

    Returns dict with keys per model: {'samples': DataFrame, 'curves': ndarray}.
    """
    # --- sample generation (paired across models) ---
    seed = stable_seed(scenario, country)
    rng  = np.random.default_rng(seed)
    gs   = rng.uniform(G_MIN, g_cap, N_SAMPLES)
    if row_type == 'A':
        thi = target_hi_common_for(S0, C_scenario, g_cap)
        if not np.isfinite(thi) or thi <= 0.001:
            # degenerate: no Type A prior support
            targets = np.full(N_SAMPLES, np.nan)
        else:
            targets = rng.uniform(0.001, thi, N_SAMPLES)
    else:  # B_fixed
        targets = np.full(N_SAMPLES, target_fixed)

    out = {}
    for model in ('Logistic', 'Gompertz'):
        accepted = []
        curves = []
        for j in range(N_SAMPLES):
            g, tgt = float(gs[j]), float(targets[j])
            if not np.isfinite(tgt):
                continue
            sol = solve_C_for_target(model, S0, g, tgt, C_scenario)
            if sol is None:
                continue
            C_s, r, tn, tp, rates, cum = sol
            if not (TP_MIN < tp < TP_MAX):
                continue
            accepted.append({
                'sample_idx': j, 'g': g, 'target': tgt, 'C_sampled': C_s,
                'r': r, 'tn': tn, 'tp': tp,
                'rate_2050': float(rates[IDX_2050]),
                'rate_2100': float(rates[IDX_2100]),
                # True model cumulative state at 2100 (exact integral, not Riemann sum)
                'cumulative_2100': float(cum[IDX_2100]),
            })
            curves.append(rates)

        samples_df = pd.DataFrame(accepted)
        curves_arr = np.array(curves) if curves else np.empty((0, N_YEARS))
        out[model] = {'samples': samples_df, 'curves': curves_arr}
    return out


print('MC engine defined.')


MC engine defined.


### Run MC across all 80 (scenario, country) combinations

Two accepted pools per combination (Logistic and Gompertz) give 160 pools total.
Expected runtime: 2 to 5 minutes on a modern machine.


In [5]:
pool_store = {}           # (scenario, country, model) -> {'samples', 'curves'}
acceptance_rows = []
sample_blocks = {s: [] for s in SCENARIO_FILES}

for _, row in scenario_country_table.iterrows():
    scenario = row['Scenario']
    country  = row['Country']
    result = mc_pool(
        scenario, country,
        C_scenario=row['C_scenario'], S0=row['S_0'],
        g_cap=row['g_cap'], row_type=row['Type'],
        target_fixed=row['target_fixed'],
    )
    for model, pool in result.items():
        pool_store[(scenario, country, model)] = pool
        n_acc = len(pool['samples'])
        acceptance_rows.append({
            'Scenario': scenario, 'Country': country, 'Model': model,
            'Type': row['Type'], 'N_attempted': N_SAMPLES,
            'N_accepted': n_acc, 'acceptance_rate': n_acc / N_SAMPLES,
            'C_scenario': row['C_scenario'], 'S_0': row['S_0'],
            'g_cap': row['g_cap'],
        })
        if n_acc:
            sd = pool['samples'].copy()
            sd.insert(0, 'Model', model)
            sd.insert(0, 'Country', country)
            sample_blocks[scenario].append(sd)
    type_tag = 'B' if row['Type'] == 'B_fixed' else 'A'
    n_L = len(result['Logistic']['samples'])
    n_G = len(result['Gompertz']['samples'])
    print(f'  [{type_tag}] {scenario:10s} {country:12s}  '
          f'L={n_L:5d}  G={n_G:5d}')

acceptance_df = pd.DataFrame(acceptance_rows)
acceptance_df.to_csv(OUT_DIR / 'acceptance.csv', index=False)

for scenario, parts in sample_blocks.items():
    if parts:
        pd.concat(parts, ignore_index=True).to_csv(
            OUT_DIR / f'samples_{scenario}.csv', index=False)

print(f'\nMC pools built. Total accepted rows: '
      f'{acceptance_df["N_accepted"].sum():,}')
print(f'Saved: acceptance.csv, samples_{{scenario}}.csv')


  [A] reference  UK            L= 1932  G= 2544


  [A] reference  US            L= 2102  G= 2569


  [A] reference  EU            L= 2122  G= 2667


  [A] reference  China         L= 1303  G= 2574


  [A] reference  Middle East   L= 1997  G= 2558


...

  [B] ipcc_high  Canada        L= 1648  G= 2687


  [B] ipcc_high  Indonesia     L=    0  G=    0


  [A] ipcc_high  Thailand      L= 1470  G= 2617


  [B] ipcc_high  Brazil        L=  699  G=    0



MC pools built. Total accepted rows: 316,029
Saved: acceptance.csv, samples_{scenario}.csv


## Section 3: Uncertainty bands

Per r4 §4 Step 3: pointwise accepted-sample percentiles
(`p5, p25, p50, p75, p95`) for each `(scenario, country, model)`.


In [6]:
band_rows = []
for (scenario, country, model), pool in pool_store.items():
    curves = pool['curves']
    if curves.size == 0:
        continue
    p5, p25, p50, p75, p95 = np.percentile(curves, [5, 25, 50, 75, 95], axis=0)
    for i, yr in enumerate(YEARS):
        band_rows.append({
            'Scenario': scenario, 'Country': country, 'Model': model,
            'Year': int(yr),
            'rate_p5':  float(p5[i]),  'rate_p25': float(p25[i]),
            'rate_p50': float(p50[i]), 'rate_p75': float(p75[i]),
            'rate_p95': float(p95[i]),
        })

bands_df = pd.DataFrame(band_rows)
bands_df.to_csv(OUT_DIR / 'bands.csv', index=False)
print(f'Bands: {len(bands_df):,} rows over {len(pool_store)} pools.')
print('Saved: bands.csv')


Bands: 22,350 rows over 160 pools.
Saved: bands.csv


## Section 4: Deterministic central pathway

Per r4 §4 Step 4: fix `C = C_scenario`, anchor to `target_anchor`, brentq
for `g*_m` per model.

- Type A: `target_anchor = median(target_joint_accepted)` across
  samples accepted by both models.
- Type B-fixed: `target_anchor = target_fixed` directly.

Fallbacks logged to `feasibility.csv`:

- `empty_pool`: joint accepted set is empty (Type A) or single-model
  pool is empty (Type B-fixed);
- `infeasible_unreachable_target`: brentq cannot bracket `g*_m` in
  `[G_MIN, g_cap]`.


In [7]:
def brentq_g_for_target(model, C_scenario, S0, target_anchor, g_cap):
    """Brent's method for g such that rate_m(2050 | C_scenario, S0, g) = target_anchor."""
    def residual(g):
        try:
            _, _, _, rates, _ = curve_for_gC(model, C_scenario, S0, T_BASE, g)
        except (ValueError, ZeroDivisionError):
            return float('nan')
        return float(rates[IDX_2050]) - target_anchor

    f_lo = residual(G_MIN)
    f_hi = residual(g_cap)
    if not (np.isfinite(f_lo) and np.isfinite(f_hi)):
        return None, 'nonfinite_residual'
    if f_lo * f_hi > 0:
        return None, f'no_bracket  f_lo={f_lo:.3e}  f_hi={f_hi:.3e}'
    try:
        g_sol = brentq(residual, G_MIN, g_cap, xtol=BRENTQ_XTOL, rtol=1e-12,
                       maxiter=200)
    except Exception as exc:
        return None, f'brentq_failed: {exc}'
    return g_sol, None


central_rows = []
timeseries_rows = []
feasibility_rows = []

for _, row in scenario_country_table.iterrows():
    scenario, country = row['Scenario'], row['Country']
    C_sc, S0 = row['C_scenario'], row['S_0']
    g_cap = row['g_cap']
    row_type = row['Type']

    # Resolve target_anchor
    pool_L = pool_store[(scenario, country, 'Logistic')]['samples']
    pool_G = pool_store[(scenario, country, 'Gompertz')]['samples']

    if row_type == 'A':
        # Joint accepted set = samples accepted by BOTH models
        joint_idx = set(pool_L['sample_idx']) & set(pool_G['sample_idx']) if (
            len(pool_L) and len(pool_G)) else set()
        if not joint_idx:
            feasibility_rows.append({
                'Scenario': scenario, 'Country': country,
                'flag': 'empty_joint_pool',
                'reason': 'Type A joint accepted set empty',
            })
            continue
        targets_joint = pool_L[pool_L['sample_idx'].isin(joint_idx)]['target'].to_numpy()
        target_anchor = float(np.median(targets_joint))
    else:  # B_fixed: log per-model empty pools (P2 fix)
        if len(pool_L) == 0:
            feasibility_rows.append({
                'Scenario': scenario, 'Country': country, 'Model': 'Logistic',
                'flag': 'empty_model_pool',
                'reason': 'Type B-fixed: Logistic MC pool empty; target unreachable under filter',
                'target_anchor': row['target_fixed'],
            })
        if len(pool_G) == 0:
            feasibility_rows.append({
                'Scenario': scenario, 'Country': country, 'Model': 'Gompertz',
                'flag': 'empty_model_pool',
                'reason': 'Type B-fixed: Gompertz MC pool empty; target unreachable under filter',
                'target_anchor': row['target_fixed'],
            })
        target_anchor = row['target_fixed']

    # Solve per-model g*_m
    per_model = {}
    for model in ('Logistic', 'Gompertz'):
        g_star, err = brentq_g_for_target(model, C_sc, S0, target_anchor, g_cap)
        if g_star is None:
            feasibility_rows.append({
                'Scenario': scenario, 'Country': country, 'Model': model,
                'flag': 'infeasible_unreachable_target',
                'reason': err, 'target_anchor': target_anchor,
            })
            continue
        try:
            r, tn, tp, rates, cum = curve_for_gC(model, C_sc, S0, T_BASE, g_star)
        except (ValueError, ZeroDivisionError) as exc:
            feasibility_rows.append({
                'Scenario': scenario, 'Country': country, 'Model': model,
                'flag': 'math_error', 'reason': str(exc),
            })
            continue
        per_model[model] = {
            'g_star': g_star, 'r': r, 'tn': tn, 'tp': tp,
            'rates': rates, 'cum': cum,
        }
        for i, yr in enumerate(YEARS):
            timeseries_rows.append({
                'Scenario': scenario, 'Country': country, 'Model': model,
                'Year': int(yr),
                'rate_central': float(rates[i]),
                # True model cumulative state (exact), not Riemann sum
                'cumulative_central': float(cum[i]),
            })

    if 'Logistic' in per_model and 'Gompertz' in per_model:
        L, G = per_model['Logistic'], per_model['Gompertz']
        central_rows.append({
            'Scenario': scenario, 'Country': country, 'Type': row_type,
            'C_scenario': C_sc, 'S_0': S0, 'g_cap': g_cap,
            'target_anchor': target_anchor,
            'g_star_L': L['g_star'], 'g_star_G': G['g_star'],
            'tn_L': L['tn'], 'tn_G': G['tn'],
            'tp_L': L['tp'], 'tp_G': G['tp'],
            'r_L': L['r'],   'r_G': G['r'],
            'rate_2050_L': float(L['rates'][IDX_2050]),
            'rate_2050_G': float(G['rates'][IDX_2050]),
            'rate_2100_L': float(L['rates'][IDX_2100]),
            'rate_2100_G': float(G['rates'][IDX_2100]),
            'g_gap': L['g_star'] - G['g_star'],
        })

scalars_central_df = pd.DataFrame(central_rows)
timeseries_central_df = pd.DataFrame(timeseries_rows)
feasibility_df = pd.DataFrame(feasibility_rows)

scalars_central_df.to_csv(OUT_DIR / 'scalars_central.csv', index=False)
timeseries_central_df.to_csv(OUT_DIR / 'timeseries_central.csv', index=False)
feasibility_df.to_csv(OUT_DIR / 'feasibility.csv', index=False)

print(f'Centrals: {len(scalars_central_df)} feasible, '
      f'{len(feasibility_df)} flagged infeasible events.')
print('Saved: scalars_central.csv, timeseries_central.csv, feasibility.csv')


Centrals: 74 feasible, 21 flagged infeasible events.
Saved: scalars_central.csv, timeseries_central.csv, feasibility.csv


## Section 5: Sensitivity analysis

Step 5a, Saltelli-based sensitivity with explicit feasibility handling
(P1c fix): Earlier drafts used mean-imputation to fill Y at infeasible
Saltelli points, which silently pulled variance toward the feasible mean.
This workflow reports two distinct Sobol results per `(scenario, country, model,
output)`:

1. `S_feasibility`: Saltelli Sobol with
   `Y_feasibility ∈ {0, 1}` (did the inverse solve?). Indices describe
   *which input drives feasibility itself*. Always defined on the full
   prior.
2. `S_conditional`: Saltelli indices computed only on the feasible
   subset (no imputation). Indices describe what drives `Y` *within*
   the feasible region. The column `feasibility_fraction` records how
   large that subset is so the reader can weight interpretation.

For Type B-fixed, target uncertainty is zero by design, so we still record
rows with `S_g = 1, S_target = 0` for completeness but skip the Saltelli
run (no sensible 2-D decomposition to compute).

Step 5b, ANOVA partial R² on accepted sets (D7): independent of
Sobol; decomposes variance of feasible outputs via OLS with interaction
and quadratic terms.


In [8]:
from SALib.sample import saltelli
from SALib.analyze import sobol as sobol_analyze


def _run_saltelli(problem, Y):
    """Wrap SALib's analyzer; return 5-tuple of floats or None if ill-posed."""
    if Y.size < 4 or not np.isfinite(Y).all():
        return None
    if np.var(Y) == 0:
        return None
    try:
        Si = sobol_analyze.analyze(
            problem, Y, calc_second_order=False, print_to_console=False,
        )
    except Exception:
        return None
    return (float(Si['S1'][0]), float(Si['S1'][1]),
            float(Si['ST'][0]), float(Si['ST'][1]),
            float(Si['ST'][0] + Si['ST'][1] - Si['S1'][0] - Si['S1'][1]))


def sobol_for_combo(scenario, country, model, row):
    """Two Sobol analyses per combination (no imputation):

    (a) feasibility Sobol: run on the full prior with binary
        `Y_feasibility ∈ {0, 1}`. Answers "which input drives whether
        the inverse has a solution inside [C_lo, C_scenario]?"
    (b) conditional Sobol: compute sensitivity of `Y ∈
        {tp, rate_2100, C_sampled}` *only when the full Saltelli design
        is feasible*. No imputation is ever used. If any prior samples are
        infeasible, conditional rows are emitted as skipped/NaN and
        interpretation falls back to feasibility Sobol + ANOVA (§5b).

    Type B-fixed is degenerate (target variance = 0); we skip Saltelli and
    emit a single row with `S_g = 1, S_target = 0` per output.
    """
    C_sc, S0, g_cap = row['C_scenario'], row['S_0'], row['g_cap']
    row_type = row['Type']
    rows_out = []

    if row_type != 'A':
        for name in ('tp', 'rate_2100', 'C_sampled'):
            rows_out.append({
                'Scenario': scenario, 'Country': country, 'Model': model,
                'Type': 'B_fixed', 'Output_Y': name,
                'kind': 'conditional_trivial',
                'S_g': 1.0, 'S_target': 0.0,
                'S_g_total': 1.0, 'S_target_total': 0.0, 'S_gt': 0.0,
                'feasibility_fraction': float('nan'),
                'n_saltelli_total': 0, 'n_saltelli_feasible': 0,
            })
        return rows_out

    thi = target_hi_common_for(S0, C_sc, g_cap)
    if not np.isfinite(thi) or thi <= 0.001:
        return []

    problem = {
        'num_vars': 2,
        'names': ['g', 'target'],
        'bounds': [[G_MIN, g_cap], [0.001, thi]],
    }
    param_values = saltelli.sample(problem, N_SOBOL_BASE,
                                   calc_second_order=False)
    N_total = len(param_values)
    Y_tp    = np.full(N_total, np.nan)
    Y_r2100 = np.full(N_total, np.nan)
    Y_C     = np.full(N_total, np.nan)
    Y_feas  = np.zeros(N_total, dtype=float)

    for k, (g, tgt) in enumerate(param_values):
        sol = solve_C_for_target(model, S0, g, tgt, C_sc)
        if sol is None:
            continue
        C_s, _, _, tp, rates, _ = sol
        Y_feas[k]  = 1.0
        Y_tp[k]    = tp
        Y_r2100[k] = float(rates[IDX_2100])
        Y_C[k]     = C_s

    N_feas = int(Y_feas.sum())
    frac   = N_feas / N_total if N_total else 0.0

    # --- (a) Feasibility Sobol: full prior, binary Y ---
    si_feas = _run_saltelli(problem, Y_feas)
    if si_feas is not None:
        s1g, s1t, stg, stt, s_int = si_feas
        rows_out.append({
            'Scenario': scenario, 'Country': country, 'Model': model,
            'Type': 'A', 'Output_Y': 'feasibility',
            'kind': 'feasibility',
            'S_g': s1g, 'S_target': s1t,
            'S_g_total': stg, 'S_target_total': stt, 'S_gt': s_int,
            'feasibility_fraction': frac,
            'n_saltelli_total': N_total, 'n_saltelli_feasible': N_feas,
        })

    # --- (b) Conditional Sobol: exact only, never imputed ---
    # Saltelli's index formula assumes the full tensor-product design.
    # We therefore compute conditional Sobol only when all prior samples
    # are feasible; otherwise we emit NaN rows and fall back to
    # feasibility Sobol + ANOVA (§5b).
    mask = np.isfinite(Y_tp)
    for name, Y in (('tp', Y_tp), ('rate_2100', Y_r2100), ('C_sampled', Y_C)):
        if mask.sum() < 8 or np.nanvar(Y[mask]) == 0:
            continue
        if frac < 1.0:
            rows_out.append({
                'Scenario': scenario, 'Country': country, 'Model': model,
                'Type': 'A', 'Output_Y': name,
                'kind': 'conditional_skipped_partial_feasibility',
                'S_g': float('nan'), 'S_target': float('nan'),
                'S_g_total': float('nan'), 'S_target_total': float('nan'),
                'S_gt': float('nan'),
                'feasibility_fraction': frac,
                'n_saltelli_total': N_total, 'n_saltelli_feasible': N_feas,
            })
            continue
        si = _run_saltelli(problem, Y)
        if si is None:
            continue
        s1g, s1t, stg, stt, s_int = si
        rows_out.append({
            'Scenario': scenario, 'Country': country, 'Model': model,
            'Type': 'A', 'Output_Y': name,
            'kind': 'conditional_exact',
            'S_g': s1g, 'S_target': s1t,
            'S_g_total': stg, 'S_target_total': stt, 'S_gt': s_int,
            'feasibility_fraction': frac,
            'n_saltelli_total': N_total, 'n_saltelli_feasible': N_feas,
        })
    return rows_out


print('Running Saltelli Sobol across 160 (scenario, country, model) combinations...')
sobol_rows = []
for _, row in scenario_country_table.iterrows():
    for model in ('Logistic', 'Gompertz'):
        sobol_rows.extend(sobol_for_combo(row['Scenario'], row['Country'], model, row))

sobol_df = pd.DataFrame(sobol_rows)
sobol_df.to_csv(OUT_DIR / 'sobol_indices.csv', index=False)
print(f'Saved: sobol_indices.csv  ({len(sobol_df)} rows)')

# Summary: kind breakdown
print('\nSobol row kinds:')
print(sobol_df['kind'].value_counts().to_string())
print('\nFeasibility fraction summary (Type A):')
sub = sobol_df[(sobol_df['kind'] == 'feasibility')]
if not sub.empty:
    print(sub['feasibility_fraction'].describe().round(3).to_string())


Running Saltelli Sobol across 160 (scenario, country, model) combinations...


/tmp/ipykernel_81287/3868472447.py:63: DeprecationWarning: `salib.sample.saltelli` will be removed in SALib 1.5.1 Please use `salib.sample.sobol`
  param_values = saltelli.sample(problem, N_SOBOL_BASE,


Saved: sobol_indices.csv  (604 rows)

Sobol row kinds:
kind
conditional_skipped_partial_feasibility    372
feasibility                                124
conditional_trivial                        108

Feasibility fraction summary (Type A):
count    124.000
mean       0.288
std        0.076
min        0.104
25%        0.224
50%        0.284
75%        0.331
max        0.492


In [9]:
# --- ANOVA partial R² on the accepted set (D7) ---

def anova_partial_r2(pool_samples, terms):
    """Compute partial R² for each term in `terms` against outputs tp, rate_2100, C_sampled.

    Uses Type-II ANOVA (reduction in residual variance when term is added).
    """
    if len(pool_samples) < 20:
        return []
    out = []
    for Y_name in ('tp', 'rate_2100', 'C_sampled'):
        Y = pool_samples[Y_name].to_numpy()
        if not np.isfinite(Y).all() or np.var(Y) == 0:
            continue
        # Build full design matrix
        design = {}
        for term in terms:
            if term == 'g':
                design['g'] = pool_samples['g'].to_numpy()
            elif term == 'target':
                design['target'] = pool_samples['target'].to_numpy()
            elif term == 'g:target':
                design['g:target'] = (pool_samples['g'] * pool_samples['target']).to_numpy()
            elif term == 'g2':
                design['g2'] = pool_samples['g'].to_numpy() ** 2
            elif term == 'target2':
                design['target2'] = pool_samples['target'].to_numpy() ** 2
        X_full = np.column_stack([design[t] for t in terms])
        X_full = np.column_stack([np.ones(len(X_full)), X_full])  # intercept
        coef_full, res_full, *_ = np.linalg.lstsq(X_full, Y, rcond=None)
        ss_res_full = np.sum((Y - X_full @ coef_full) ** 2)
        ss_tot = np.sum((Y - Y.mean()) ** 2)
        r2_full = 1 - ss_res_full / ss_tot if ss_tot > 0 else np.nan

        # Partial R²: remove each term
        partials = {}
        for k, t in enumerate(terms, start=1):
            cols = [0] + [i for i in range(1, X_full.shape[1]) if i != k]
            X_red = X_full[:, cols]
            coef_red, *_ = np.linalg.lstsq(X_red, Y, rcond=None)
            ss_res_red = np.sum((Y - X_red @ coef_red) ** 2)
            partials[f'partial_R2_{t}'] = float((ss_res_red - ss_res_full) / ss_tot) if ss_tot > 0 else np.nan

        out.append({'Output_Y': Y_name, 'R2_full': float(r2_full),
                    'n_accepted': len(pool_samples), **partials})
    return out


anova_rows = []
for _, row in scenario_country_table.iterrows():
    scenario, country, row_type = row['Scenario'], row['Country'], row['Type']
    for model in ('Logistic', 'Gompertz'):
        samples = pool_store[(scenario, country, model)]['samples']
        if len(samples) < 20:
            continue
        terms = (['g', 'target', 'g:target', 'g2', 'target2']
                 if row_type == 'A' else ['g', 'g2'])
        for res in anova_partial_r2(samples, terms):
            anova_rows.append({
                'Scenario': scenario, 'Country': country, 'Model': model,
                'Type': row_type, **res,
            })

anova_df = pd.DataFrame(anova_rows)
anova_df.to_csv(OUT_DIR / 'anova_conditional.csv', index=False)
print(f'Saved: anova_conditional.csv  ({len(anova_df)} rows)')


Saved: anova_conditional.csv  (447 rows)


## Section 6: Bootstrap global aggregation

Per r4 §4 Step 7 with `B = 1000`. For each `(scenario, model)`: draw one
accepted sample path per country with a non-empty pool, sum, repeat.

Deterministic global central = sum of per-country `(C_scenario, g*_m)` central lines.

Completeness tracking (P1b fix): every output row records which
countries were included and which were excluded (due to empty MC pool or
missing central). The `is_complete` flag is `True` only when all 10
countries contribute. `ipcc_low` and `ipcc_high` in particular are
expected to be incomplete; Fig 4 annotates these explicitly so the
reader never reads an incomplete aggregate as a full-scenario total.


In [10]:
rng_global = np.random.default_rng(20260422)
ALL_COUNTRIES = sorted({c for df in scenario_csvs.values() for c in df['Country']})
N_ALL = len(ALL_COUNTRIES)

global_bands_rows = []
global_central_rows = []

for scenario in SCENARIO_FILES:
    scenario_countries = list(scenario_csvs[scenario]['Country'])
    for model in ('Logistic', 'Gompertz'):
        # --- Collect accepted curves per country, track exclusions (P1b) ---
        country_curves = {}
        excluded_mc = []   # countries dropped from bootstrap sum
        for country in scenario_countries:
            curves = pool_store[(scenario, country, model)]['curves']
            if curves.size > 0:
                country_curves[country] = curves
            else:
                excluded_mc.append(country)
        n_included_mc = len(country_curves)

        # --- Central completeness (separate check: some centrals infeasible) ---
        sub_det = timeseries_central_df[
            (timeseries_central_df['Scenario'] == scenario) &
            (timeseries_central_df['Model']    == model)
        ]
        countries_with_central = set(sub_det['Country'].unique())
        excluded_central = [c for c in scenario_countries
                            if c not in countries_with_central]
        n_included_central = len(countries_with_central)

        is_complete_mc      = (n_included_mc      == len(scenario_countries))
        is_complete_central = (n_included_central == len(scenario_countries))

        # --- Bootstrap ---
        if country_curves:
            global_traj = np.zeros((B_BOOTSTRAP, N_YEARS))
            for b in range(B_BOOTSTRAP):
                total = np.zeros(N_YEARS)
                for country, curves in country_curves.items():
                    pick = rng_global.integers(0, len(curves))
                    total += curves[pick]
                global_traj[b] = total
            p5, p25, p50, p75, p95 = np.percentile(
                global_traj, [5, 25, 50, 75, 95], axis=0)
        else:
            p5 = p25 = p50 = p75 = p95 = np.zeros(N_YEARS)

        # --- Deterministic global central ---
        if not sub_det.empty:
            global_det = sub_det.groupby('Year')['rate_central'].sum().reset_index()
            det_by_year = dict(zip(global_det['Year'], global_det['rate_central']))
        else:
            det_by_year = {}

        excluded_mc_str      = ';'.join(excluded_mc)      if excluded_mc      else ''
        excluded_central_str = ';'.join(excluded_central) if excluded_central else ''

        for i, yr in enumerate(YEARS):
            global_bands_rows.append({
                'Scenario': scenario, 'Model': model, 'Year': int(yr),
                'rate_p5':  float(p5[i]),  'rate_p25': float(p25[i]),
                'rate_p50': float(p50[i]), 'rate_p75': float(p75[i]),
                'rate_p95': float(p95[i]),
                'n_countries_included': n_included_mc,
                'n_countries_total': len(scenario_countries),
                'countries_excluded': excluded_mc_str,
                'is_complete': is_complete_mc,
            })
            global_central_rows.append({
                'Scenario': scenario, 'Model': model,
                'Year': int(yr),
                'rate_central_global': float(det_by_year.get(int(yr), 0.0)),
                'n_countries_included': n_included_central,
                'n_countries_total': len(scenario_countries),
                'countries_excluded': excluded_central_str,
                'is_complete': is_complete_central,
            })

global_bands_df = pd.DataFrame(global_bands_rows)
global_central_df = pd.DataFrame(global_central_rows)
global_bands_df.to_csv(OUT_DIR / 'global_bands.csv', index=False)
global_central_df.to_csv(OUT_DIR / 'global_central.csv', index=False)
print(f'Global bands: {len(global_bands_df):,} rows')
print(f'Global centrals: {len(global_central_df):,} rows')

# Completeness summary
incomplete_mc = global_bands_df[~global_bands_df['is_complete']][
    ['Scenario','Model','n_countries_included','n_countries_total','countries_excluded']
].drop_duplicates()
if not incomplete_mc.empty:
    print('\nIncomplete MC aggregations:')
    print(incomplete_mc.to_string(index=False))
incomplete_det = global_central_df[~global_central_df['is_complete']][
    ['Scenario','Model','n_countries_included','n_countries_total','countries_excluded']
].drop_duplicates()
if not incomplete_det.empty:
    print('\nIncomplete deterministic centrals:')
    print(incomplete_det.to_string(index=False))
print('Saved: global_bands.csv, global_central.csv')


Global bands: 2,400 rows
Global centrals: 2,400 rows

Incomplete MC aggregations:
 Scenario    Model  n_countries_included  n_countries_total     countries_excluded
 ipcc_low Logistic                     7                 10       US;Canada;Brazil
 ipcc_low Gompertz                     7                 10       US;Canada;Brazil
ipcc_high Logistic                     8                 10        China;Indonesia
ipcc_high Gompertz                     7                 10 China;Indonesia;Brazil

Incomplete deterministic centrals:
 Scenario    Model  n_countries_included  n_countries_total     countries_excluded
 ipcc_low Logistic                     8                 10              US;Brazil
 ipcc_low Gompertz                     7                 10       US;Canada;Brazil
ipcc_high Logistic                     8                 10        China;Indonesia
ipcc_high Gompertz                     7                 10 China;Indonesia;Brazil
Saved: global_bands.csv, global_central.csv


## Section 7: Screening handoff preparation

Per r4 §4 Step 8. Three kinds of trajectory outputs for the
`02_co2block_screening` module:

1. Deterministic central, already saved as `timeseries_central.csv`.
2. Robustness sample paths: `K = 200` accepted paths per `(s, i, m)`,
   each a real self-consistent trajectory. For screening batch mode.
3. Quartile-representative sample paths: single accepted sample
   closest to q25 and q75 of `cumulative_2100` (D6, more stable than
   pointwise bands, closer to policy interpretation).


In [11]:
robustness_rows = []
q25_q75_rows   = []

rng_screen = np.random.default_rng(20260422 + 1)

for (scenario, country, model), pool in pool_store.items():
    samples = pool['samples']
    curves  = pool['curves']
    n = len(samples)
    if n == 0:
        continue

    # --- Robustness subsample K ---
    K_eff = min(K_SCREEN, n)
    sel = rng_screen.choice(n, size=K_eff, replace=False)
    for k, idx in enumerate(sel):
        row = samples.iloc[idx]
        curve = curves[idx]
        for i, yr in enumerate(YEARS):
            robustness_rows.append({
                'Scenario': scenario, 'Country': country, 'Model': model,
                'k_index': k, 'sample_idx': int(row['sample_idx']),
                'Year': int(yr),
                'rate': float(curve[i]),
            })

    # --- Quartile-representative paths (by cumulative_2100) ---
    S_values = samples['cumulative_2100'].to_numpy()
    q25, q75 = np.percentile(S_values, [25, 75])
    j_q25 = int(np.argmin(np.abs(S_values - q25)))
    j_q75 = int(np.argmin(np.abs(S_values - q75)))
    for label, j in (('q25', j_q25), ('q75', j_q75)):
        row = samples.iloc[j]
        curve = curves[j]
        for i, yr in enumerate(YEARS):
            q25_q75_rows.append({
                'Scenario': scenario, 'Country': country, 'Model': model,
                'quartile': label, 'sample_idx': int(row['sample_idx']),
                'g': float(row['g']), 'target': float(row['target']),
                'C_sampled': float(row['C_sampled']),
                'cumulative_2100_sample': float(row['cumulative_2100']),
                'cumulative_2100_quartile_ref': float(q25 if label == 'q25' else q75),
                'Year': int(yr),
                'rate': float(curve[i]),
            })

# Robustness file is very big (K=200 × 160 × 150 years); compress columns
robustness_df = pd.DataFrame(robustness_rows)
q25_q75_df    = pd.DataFrame(q25_q75_rows)
robustness_df.to_csv(OUT_DIR / 'robustness_sample_paths.csv', index=False)
q25_q75_df.to_csv(OUT_DIR / 'q25_q75_sample_paths.csv', index=False)
print(f'Robustness paths: {len(robustness_df):,} rows '
      f'(K={K_SCREEN} × 160 pools × {N_YEARS} years)')
print(f'Quartile paths:   {len(q25_q75_df):,} rows '
      f'(2 quartiles × 160 pools × {N_YEARS} years)')
print('Saved: robustness_sample_paths.csv, q25_q75_sample_paths.csv')


Robustness paths: 4,470,000 rows (K=200 × 160 pools × 150 years)
Quartile paths:   44,700 rows (2 quartiles × 160 pools × 150 years)
Saved: robustness_sample_paths.csv, q25_q75_sample_paths.csv


## Section 8: Main-text figures

Per r4 §3.6. Four main-text figures:

- Fig 1: representative country pathway profiles
- Fig 2: cross-model deterministic comparison
- Fig 3: Sobol variance decomposition for `rate_2100`
- Fig 4: global aggregation


In [12]:
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Helvetica', 'Arial', 'DejaVu Sans'],
    'font.size': 9,
    'axes.titlesize': 11, 'axes.titleweight': 'bold',
    'axes.labelsize': 9,
    'xtick.labelsize': 8, 'ytick.labelsize': 8,
    'legend.fontsize': 8,
    'axes.spines.top': False, 'axes.spines.right': False,
    'pdf.fonttype': 42, 'ps.fonttype': 42,
})

COLOR = {
    'Logistic': {'band_hi': '#4e79a7', 'band_lo': '#aec7e8', 'central': '#1f4e79'},
    'Gompertz': {'band_hi': '#e15759', 'band_lo': '#f4a8a7', 'central': '#7a1d1d'},
}
CLR_REF        = '#222'
TN_LINE        = '#2c7873'   # teal, inflection year (dotted)
TP_LINE        = '#1f3a68'   # navy, peak-rate year (dash-dot)
CHECKPOINT_CLR = '#c8c8c8'   # light grey, 2050 / 2100 policy checkpoints

PROFILE_LEGEND_HANDLES = [
    Line2D([0], [0], color=CLR_REF, lw=2.0, label='Central pathway'),
    Line2D([0], [0], color=COLOR['Logistic']['band_hi'], lw=6, alpha=0.55,
           label='IQR (p25\u2013p75)'),
    Line2D([0], [0], color=TN_LINE, lw=1.3, ls=':',
           label='Inflection year $t_n$  (rate decelerates)'),
    Line2D([0], [0], color=TP_LINE, lw=1.3, ls='-.',
           label='Peak-rate year $t_p$  (maximum annual rate)'),
    Line2D([0], [0], color=CHECKPOINT_CLR, lw=0.8, ls='--',
           label='Policy checkpoints (2050, 2100)'),
]


def _sample_curve_subset(pool, n_plot=150):
    # Return the exact subset of sample curves plotted in a profile panel.
    if pool is None or pool['curves'].size == 0:
        return np.empty((0, N_YEARS))
    curves = pool['curves']
    n_plot = min(n_plot, len(curves))
    rng_s = np.random.default_rng(42)
    idxs = rng_s.choice(len(curves), size=n_plot, replace=False)
    return curves[idxs]


def pair_profile_ymax(scenario, country):
    # Common y-axis max for the (country, scenario) Logistic/Gompertz pair.
    # y_max still accommodates the scatter curve max (which can exceed IQR)
    # so the ensemble stays visible even though the 90% CI band is no longer drawn.
    vals = []
    for model in ('Logistic', 'Gompertz'):
        sub_b = bands_df[(bands_df['Scenario'] == scenario) &
                         (bands_df['Country']  == country)  &
                         (bands_df['Model']    == model)]
        sub_t = timeseries_central_df[
            (timeseries_central_df['Scenario'] == scenario) &
            (timeseries_central_df['Country']  == country)  &
            (timeseries_central_df['Model']    == model)
        ]
        pool = pool_store.get((scenario, country, model))
        if not sub_b.empty:
            vals.append(float(sub_b['rate_p95'].max()) * 1000.0)
        if not sub_t.empty:
            vals.append(float(sub_t['rate_central'].max()) * 1000.0)
        curves_sub = _sample_curve_subset(pool)
        if curves_sub.size:
            vals.append(float(curves_sub.max()) * 1000.0)
    if not vals:
        return 1.0
    ymax = max(vals)
    return 1.08 * ymax if ymax > 0 else 1.0


def central_metrics_for_panel(scenario, country, model):
    # Return central metrics used for annotation in a profile panel.
    sub = scalars_central_df[
        (scalars_central_df['Scenario'] == scenario) &
        (scalars_central_df['Country']  == country)
    ]
    if sub.empty:
        return None
    row = sub.iloc[0]
    if model == 'Logistic':
        tn = float(row['tn_L'])
        tp = float(row['tp_L'])
    else:
        tn = float(row['tn_G'])
        tp = float(row['tp_G'])

    ts = timeseries_central_df[
        (timeseries_central_df['Scenario'] == scenario) &
        (timeseries_central_df['Country']  == country)  &
        (timeseries_central_df['Model']    == model)
    ].sort_values('Year')
    if ts.empty:
        return None
    peak_idx = ts['rate_central'].idxmax()
    peak_row = ts.loc[peak_idx]
    return {
        'tn': tn,
        'tp': tp,
        'peak_year_discrete': int(peak_row['Year']),
        'peak_rate_mt_yr': float(peak_row['rate_central']) * 1000.0,
        'rate_2050_mt_yr': float(ts.loc[ts['Year'] == 2050, 'rate_central'].iloc[0]) * 1000.0,
        'rate_2100_mt_yr': float(ts.loc[ts['Year'] == 2100, 'rate_central'].iloc[0]) * 1000.0,
    }


def plot_profile(ax, scenario, country, model, show_ylabel=True, title=None,
                 y_max=None, annotate=True):
    # One panel: scatter + IQR + central line.
    # 90% CI band dropped: MC scatter overlay serves as tail-visualisation layer.
    sub_b = bands_df[(bands_df['Scenario'] == scenario) &
                     (bands_df['Country']  == country)  &
                     (bands_df['Model']    == model)].sort_values('Year')
    sub_t = timeseries_central_df[
        (timeseries_central_df['Scenario'] == scenario) &
        (timeseries_central_df['Country']  == country)  &
        (timeseries_central_df['Model']    == model)
    ].sort_values('Year')
    pool = pool_store.get((scenario, country, model))
    c = COLOR[model]

    # Scatter of ~150 accepted MC sample curves (tail layer)
    curves_sub = _sample_curve_subset(pool)
    if curves_sub.size:
        for curve in curves_sub:
            ax.plot(YEARS, curve * 1000, color=c['central'],
                    alpha=0.03, lw=0.35, zorder=1)

    # IQR band (only)
    if not sub_b.empty:
        yrs = sub_b['Year'].to_numpy()
        p25 = sub_b['rate_p25'].to_numpy() * 1000
        p75 = sub_b['rate_p75'].to_numpy() * 1000
        ax.fill_between(yrs, p25, p75, color=c['band_hi'], alpha=0.55,
                        label='IQR', zorder=3)

    # Central pathway
    if not sub_t.empty:
        ax.plot(sub_t['Year'], sub_t['rate_central'] * 1000,
                color=CLR_REF, lw=2.0, label='central', zorder=5)

    # 2050 / 2100 policy checkpoints (behind the model-derived feature lines)
    ax.axvline(2050, color=CHECKPOINT_CLR, ls='--', lw=0.6, alpha=0.9, zorder=2)
    ax.axvline(2100, color=CHECKPOINT_CLR, ls='--', lw=0.6, alpha=0.9, zorder=2)

    # Model-derived feature lines and annotation
    metrics = central_metrics_for_panel(scenario, country, model) if annotate else None
    if metrics is not None:
        ax.axvline(metrics['tn'], color=TN_LINE, ls=':',  lw=1.3, alpha=0.95, zorder=4)
        ax.axvline(metrics['tp'], color=TP_LINE, ls='-.', lw=1.3, alpha=0.95, zorder=4)
        # Monospace-aligned annotation; real newlines (no \n literal bug).
        txt = (
            f"Inflection yr:  {metrics['tn']:.1f}\n"
            f"Peak-rate yr:   {metrics['tp']:.1f}\n"
            f"Peak rate:      {metrics['peak_rate_mt_yr']:,.0f} Mt CO$_2$/yr"
        )
        ax.text(
            0.02, 0.98, txt,
            transform=ax.transAxes,
            ha='left', va='top', fontsize=7.5, family='monospace',
            bbox=dict(facecolor='white', edgecolor='#ccc', alpha=0.88,
                      boxstyle='round,pad=0.35'),
            zorder=6,
        )
    elif annotate:
        ax.text(
            0.02, 0.98, 'no central\nsolution',
            transform=ax.transAxes,
            ha='left', va='top', fontsize=7.5,
            bbox=dict(facecolor='white', edgecolor='none', alpha=0.78,
                      boxstyle='round,pad=0.25'),
            zorder=6,
        )

    ax.set_xlim(2030, T_END)
    if y_max is not None:
        ax.set_ylim(0, y_max)
    if show_ylabel:
        ax.set_ylabel('Mt CO$_2$/yr')
    ax.set_title(title or f'{country} · {scenario} · {model}', fontsize=10)


print('Figure helpers defined.')


Figure helpers defined.


In [13]:
# --- Fig 1: representative country pathway profiles ---
# Representative main-text subset, but with paired y-axis scaling so that
# Logistic and Gompertz are directly comparable within each country/scenario.
REP_COUNTRIES = ['China', 'US', 'EU', 'UK', 'Brazil']
REP_SCENARIOS = ['reference', 'policy']

fig1, axes = plt.subplots(len(REP_COUNTRIES), len(REP_SCENARIOS) * 2,
                          figsize=(14, 10), sharex=True)
for i, ctry in enumerate(REP_COUNTRIES):
    for j, scen in enumerate(REP_SCENARIOS):
        y_max = pair_profile_ymax(scen, ctry)
        for k, model in enumerate(('Logistic', 'Gompertz')):
            ax = axes[i, 2 * j + k]
            title = f'{ctry} · {scen} · {model}'
            plot_profile(ax, scen, ctry, model,
                         show_ylabel=(j == 0 and k == 0),
                         title=title,
                         y_max=y_max)
            if i < len(REP_COUNTRIES) - 1:
                ax.set_xlabel('')
            else:
                ax.set_xlabel('Year')
handles = PROFILE_LEGEND_HANDLES
labels = [h.get_label() for h in handles]
fig1.legend(handles, labels, loc='upper center', ncol=5, frameon=False,
            bbox_to_anchor=(0.5, 0.99), fontsize=8)
fig1.suptitle('Fig 1 — Representative country pathways (paired y-axis within country/scenario)',
              fontsize=13, fontweight='bold', y=1.03)
fig1.tight_layout(rect=[0, 0, 1, 0.95])
fig1.savefig(FIG_DIR / 'fig1_representative_profiles.pdf', dpi=200, bbox_inches='tight')
fig1.savefig(FIG_DIR / 'fig1_representative_profiles.png', dpi=180, bbox_inches='tight')
plt.close(fig1)
print('Saved fig1_representative_profiles.{pdf,png}')


1 extra bytes in post.stringData array


'created' timestamp seems very low; regarding as unix timestamp


Saved fig1_representative_profiles.{pdf,png}


In [14]:
# --- Fig 2: cross-model diagnostic ---
# Panel (a): g*_Logistic vs g*_Gompertz scatter (y=x reference)
# Panel (b): per-scenario violin of C_sampled / C_scenario from MC pool

fig2, (ax_a, ax_b) = plt.subplots(1, 2, figsize=(14, 6))

# Panel (a)
s = scalars_central_df
scenarios_order = ['reference','minimum','maximum','growth10',
                   'us1gt','policy','ipcc_low','ipcc_high']
cmap = plt.get_cmap('tab10')
for idx, scen in enumerate(scenarios_order):
    sub = s[s['Scenario'] == scen]
    if sub.empty:
        continue
    ax_a.scatter(sub['g_star_L'] * 100, sub['g_star_G'] * 100,
                 s=40, color=cmap(idx), alpha=0.8, label=scen)
lim_max = max(s['g_star_L'].max(), s['g_star_G'].max()) * 100 * 1.05
ax_a.plot([0, lim_max], [0, lim_max], 'k--', lw=0.8, alpha=0.5)
ax_a.set_xlabel('$g^*_\\mathrm{Logistic}$ (%)')
ax_a.set_ylabel('$g^*_\\mathrm{Gompertz}$ (%)')
ax_a.set_title('(a) Model-induced $g^*$ gap', fontweight='bold')
ax_a.legend(loc='best', fontsize=7, ncol=2)

# Panel (b): C_sampled / C_scenario violin per scenario
ratios_by_scen = {scen: [] for scen in scenarios_order}
for (scen, ctry, mdl), pool in pool_store.items():
    if scen not in ratios_by_scen:
        continue
    s_df = pool['samples']
    if len(s_df) == 0:
        continue
    # fetch C_scenario
    row = scenario_country_table[(scenario_country_table['Scenario'] == scen) &
                                 (scenario_country_table['Country']  == ctry)]
    if row.empty:
        continue
    C_sc = float(row.iloc[0]['C_scenario'])
    ratios_by_scen[scen].extend((s_df['C_sampled'] / C_sc).tolist())

positions = range(1, len(scenarios_order) + 1)
data = [ratios_by_scen[s] if ratios_by_scen[s] else [0] for s in scenarios_order]
parts = ax_b.violinplot(data, positions=positions, showmedians=True)
ax_b.axhline(1.0, color='red', ls='--', lw=0.8, alpha=0.6, label='C_sampled = C_scenario')
ax_b.set_xticks(positions)
ax_b.set_xticklabels(scenarios_order, rotation=30, ha='right')
ax_b.set_ylabel('$C_\\mathrm{sampled} / C_\\mathrm{scenario}$')
ax_b.set_title('(b) Capacity-assumption stress', fontweight='bold')
ax_b.legend(loc='best', fontsize=8)

fig2.suptitle('Fig 2 — Cross-model deterministic comparison (v8)',
              fontsize=13, fontweight='bold')
fig2.tight_layout(rect=[0, 0, 1, 0.96])
fig2.savefig(FIG_DIR / 'fig2_cross_model.pdf', dpi=200, bbox_inches='tight')
fig2.savefig(FIG_DIR / 'fig2_cross_model.png', dpi=180, bbox_inches='tight')
plt.close(fig2)
print('Saved fig2_cross_model.{pdf,png}')


1 extra bytes in post.stringData array


'created' timestamp seems very low; regarding as unix timestamp


Saved fig2_cross_model.{pdf,png}


In [15]:
# --- Fig 3: Sobol decomposition for rate_2100 ---
# Main text: stacked bars per country, per scenario.
# No imputation: show exact conditional rows for Type A and trivial rows
# for Type B-fixed only.

Y_MAIN = 'rate_2100'
countries_order = ['UK','US','EU','China','Middle East',
                   'Australia','Canada','Indonesia','Thailand','Brazil']

fig3, axes = plt.subplots(len(scenarios_order), 1, figsize=(12, 12), sharex=True)
for i, scen in enumerate(scenarios_order):
    ax = axes[i]
    sub = sobol_df[(sobol_df['Scenario'] == scen) &
                   (sobol_df['Output_Y'] == Y_MAIN) &
                   (sobol_df['Model']    == 'Logistic') &
                   (sobol_df['kind'].isin(
                       ['conditional_exact', 'conditional_trivial']))]
    if sub.empty:
        ax.set_visible(False)
        continue
    idx_map = {c: k for k, c in enumerate(countries_order)}
    xs, s_g, s_t, s_i = [], [], [], []
    for _, r in sub.iterrows():
        c = r['Country']
        if c not in idx_map:
            continue
        xs.append(idx_map[c])
        total = abs(r['S_g_total']) + abs(r['S_target_total']) + abs(r['S_gt'])
        if total == 0:
            total = 1.0
        s_g.append(abs(r['S_g']) / total)
        s_t.append(abs(r['S_target']) / total)
        s_i.append(max(0.0, abs(r['S_gt']) / total))
    s_g = np.array(s_g); s_t = np.array(s_t); s_i = np.array(s_i)
    ax.bar(xs, s_g, color='#1f77b4', label='S_g')
    ax.bar(xs, s_t, bottom=s_g, color='#ff7f0e', label='S_target')
    ax.bar(xs, s_i, bottom=s_g + s_t, color='#888', label='interaction')
    ax.set_ylabel(scen, fontsize=9)
    ax.set_ylim(0, 1.05)
    if i == 0:
        ax.legend(loc='upper right', fontsize=7)
axes[-1].set_xticks(range(len(countries_order)))
axes[-1].set_xticklabels(countries_order, rotation=30, ha='right', fontsize=8)
fig3.suptitle(f'Fig 3 — Sobol variance decomposition of {Y_MAIN} (Logistic)',
              fontsize=13, fontweight='bold')
fig3.tight_layout(rect=[0, 0, 1, 0.97])
fig3.savefig(FIG_DIR / 'fig3_sobol_rate2100.pdf', dpi=200, bbox_inches='tight')
fig3.savefig(FIG_DIR / 'fig3_sobol_rate2100.png', dpi=180, bbox_inches='tight')
plt.close(fig3)
print('Saved fig3_sobol_rate2100.{pdf,png}')


1 extra bytes in post.stringData array


'created' timestamp seems very low; regarding as unix timestamp


Saved fig3_sobol_rate2100.{pdf,png}


In [16]:
# --- Fig 4: global bootstrap aggregation ---
# P1b annotation: flag scenarios where not all 10 countries contribute.
fig4, axes = plt.subplots(2, 4, figsize=(16, 8), sharex=True)
for i, scen in enumerate(scenarios_order):
    ax = axes[i // 4, i % 4]
    title_suffix = ''
    incomplete_any = False
    for mdl in ('Logistic', 'Gompertz'):
        c = COLOR[mdl]
        sub_b = global_bands_df[(global_bands_df['Scenario'] == scen) &
                                (global_bands_df['Model']    == mdl)].sort_values('Year')
        sub_c = global_central_df[(global_central_df['Scenario'] == scen) &
                                  (global_central_df['Model']    == mdl)].sort_values('Year')
        if not sub_b.empty:
            first = sub_b.iloc[0]
            if not bool(first['is_complete']):
                incomplete_any = True
                title_suffix += (
                    f"\\n[{mdl}] incomplete: "
                    f"{first['n_countries_included']}/{first['n_countries_total']} "
                    f"({first['countries_excluded']})"
                )
            yrs = sub_b['Year']
            ax.fill_between(yrs, sub_b['rate_p5']*1000, sub_b['rate_p95']*1000,
                            color=c['band_lo'], alpha=0.35)
            ax.fill_between(yrs, sub_b['rate_p25']*1000, sub_b['rate_p75']*1000,
                            color=c['band_hi'], alpha=0.55)
        if not sub_c.empty:
            ax.plot(sub_c['Year'], sub_c['rate_central_global']*1000,
                    color=c['central'], lw=1.8, label=mdl)
    for cp in (2050, 2100):
        ax.axvline(cp, color='#888', ls='--', lw=0.7, alpha=0.4)
    title = scen + (' ⚠' if incomplete_any else '')
    ax.set_title(title, color=('#b00' if incomplete_any else 'black'))
    ax.set_xlim(2030, T_END)
    if title_suffix:
        ax.text(0.02, 0.98, title_suffix.replace(r'\n', '\n').lstrip(),
                transform=ax.transAxes, va='top', ha='left', fontsize=6,
                color='#b00',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                          edgecolor='#b00', alpha=0.85))
    if i % 4 == 0:
        ax.set_ylabel('Global Mt CO$_2$/yr')
    if i // 4 == 1:
        ax.set_xlabel('Year')
    if i == 0:
        ax.legend(loc='upper right', fontsize=8)
fig4.suptitle('Fig 4 — Global bootstrap aggregation (v8) · '
              '⚠ = not all 10 countries contribute',
              fontsize=13, fontweight='bold')
fig4.tight_layout(rect=[0, 0, 1, 0.94])
fig4.savefig(FIG_DIR / 'fig4_global_aggregation.pdf', dpi=200, bbox_inches='tight')
fig4.savefig(FIG_DIR / 'fig4_global_aggregation.png', dpi=180, bbox_inches='tight')
plt.close(fig4)
print('Saved fig4_global_aggregation.{pdf,png}')


/tmp/ipykernel_81287/3938365578.py:51: UserWarning: Glyph 9888 (\N{WARNING SIGN}) missing from font(s) Helvetica.
  fig4.tight_layout(rect=[0, 0, 1, 0.94])


/tmp/ipykernel_81287/3938365578.py:52: UserWarning: Glyph 9888 (\N{WARNING SIGN}) missing from font(s) Helvetica.
  fig4.savefig(FIG_DIR / 'fig4_global_aggregation.pdf', dpi=200, bbox_inches='tight')
1 extra bytes in post.stringData array


'created' timestamp seems very low; regarding as unix timestamp


/tmp/ipykernel_81287/3938365578.py:53: UserWarning: Glyph 9888 (\N{WARNING SIGN}) missing from font(s) Helvetica.
  fig4.savefig(FIG_DIR / 'fig4_global_aggregation.png', dpi=180, bbox_inches='tight')


Saved fig4_global_aggregation.{pdf,png}


## Section 9: Supplementary figure families

- Per-country figure1-style profiles (10 PDFs): 8 scenarios × 2 model
  columns, with a shared y-axis within each country/scenario pair so
  Logistic and Gompertz are directly comparable.
- Per-scenario dashboard (8 PDFs): 10-country small multiples.

For brevity the Sobol per-country and global-contribution families are
deferred to a follow-up notebook if needed; the main CSV outputs are
already sufficient to generate them.


In [17]:
# --- Per-country figure1-style profiles: 8 scenarios × 2 models ---
FIG_SUB = FIG_DIR / 'per_country_dossier'
FIG_SUB.mkdir(exist_ok=True)

for country in countries_order:
    fig, axes = plt.subplots(len(scenarios_order), 2,
                             figsize=(12, 2.2 * len(scenarios_order)),
                             sharex=True, sharey='row')
    if len(scenarios_order) == 1:
        axes = np.array([axes])

    for i, scen in enumerate(scenarios_order):
        y_max = pair_profile_ymax(scen, country)
        for j, model in enumerate(('Logistic', 'Gompertz')):
            ax = axes[i, j]
            title = f'{scen} · {model}'
            plot_profile(ax, scen, country, model,
                         show_ylabel=(j == 0),
                         title=title,
                         y_max=y_max)
            if j == 0:
                ax.set_ylabel(f'{scen}\nMt CO$_2$/yr')
            if i < len(scenarios_order) - 1:
                ax.set_xlabel('')
            else:
                ax.set_xlabel('Year')
            if i == 0 and j == 0:
                ax.legend(PROFILE_LEGEND_HANDLES,
                          [h.get_label() for h in PROFILE_LEGEND_HANDLES],
                          loc='upper left', fontsize=7)

    fig.suptitle(
        f'{country} — all scenarios, paired y-axis within each scenario (v8)',
        fontsize=12, fontweight='bold'
    )
    fig.tight_layout(rect=[0, 0, 1, 0.98])
    safe = country.replace(' ', '_')
    fig.savefig(FIG_SUB / f'{safe}_dossier.pdf', dpi=180, bbox_inches='tight')
    plt.close(fig)

print(f'Saved {len(countries_order)} per-country figure1-style profiles to {FIG_SUB}')


1 extra bytes in post.stringData array


'created' timestamp seems very low; regarding as unix timestamp


1 extra bytes in post.stringData array


'created' timestamp seems very low; regarding as unix timestamp


1 extra bytes in post.stringData array


...

1 extra bytes in post.stringData array


'created' timestamp seems very low; regarding as unix timestamp


1 extra bytes in post.stringData array


'created' timestamp seems very low; regarding as unix timestamp


Saved 10 per-country figure1-style profiles to /Users/xg320/Desktop/IC-Sam-works/paper/Iman-2026/01_growth_model/output/v8_2026-06-01/figures/per_country_dossier


In [18]:
# --- Per-scenario dashboard: 10 countries × (L + G overlay) ---
FIG_DASH = FIG_DIR / 'scenario_dashboards'
FIG_DASH.mkdir(exist_ok=True)

for scen in scenarios_order:
    fig, axes = plt.subplots(2, 5, figsize=(18, 7), sharex=True, sharey=False)
    for k, country in enumerate(countries_order):
        ax = axes[k // 5, k % 5]
        for model in ('Logistic', 'Gompertz'):
            c = COLOR[model]
            sub_b = bands_df[(bands_df['Scenario'] == scen) &
                             (bands_df['Country']  == country) &
                             (bands_df['Model']    == model)].sort_values('Year')
            sub_t = timeseries_central_df[
                (timeseries_central_df['Scenario'] == scen) &
                (timeseries_central_df['Country']  == country) &
                (timeseries_central_df['Model']    == model)
            ].sort_values('Year')
            if not sub_b.empty:
                ax.fill_between(sub_b['Year'], sub_b['rate_p25']*1000,
                                sub_b['rate_p75']*1000,
                                color=c['band_hi'], alpha=0.4)
            if not sub_t.empty:
                ax.plot(sub_t['Year'], sub_t['rate_central']*1000,
                        color=c['central'], lw=1.3, label=model)
        ax.axvline(2050, color='#888', ls='--', lw=0.6, alpha=0.4)
        ax.set_title(country, fontsize=9)
        if k % 5 == 0:
            ax.set_ylabel('Mt CO$_2$/yr')
        if k // 5 == 1:
            ax.set_xlabel('Year')
    fig.suptitle(f'{scen} — country dashboard (v8)', fontsize=12, fontweight='bold')
    fig.tight_layout(rect=[0, 0, 1, 0.96])
    fig.savefig(FIG_DASH / f'{scen}_dashboard.pdf', dpi=180, bbox_inches='tight')
    fig.savefig(FIG_DASH / f'{scen}_dashboard.png', dpi=180, bbox_inches='tight')
    plt.close(fig)

print(f'Saved 8 per-scenario dashboards to {FIG_DASH}')


1 extra bytes in post.stringData array


'created' timestamp seems very low; regarding as unix timestamp


1 extra bytes in post.stringData array


'created' timestamp seems very low; regarding as unix timestamp


1 extra bytes in post.stringData array


...

1 extra bytes in post.stringData array


'created' timestamp seems very low; regarding as unix timestamp


1 extra bytes in post.stringData array


'created' timestamp seems very low; regarding as unix timestamp


Saved 8 per-scenario dashboards to /Users/xg320/Desktop/IC-Sam-works/paper/Iman-2026/01_growth_model/output/v8_2026-06-01/figures/scenario_dashboards


## Summary

Outputs in `01_growth_model/output/v8_2026-06-01/`:

- Data CSVs: `acceptance.csv`, `samples_{scenario}.csv`, `bands.csv`,
  `scalars_central.csv`, `timeseries_central.csv`, `feasibility.csv`,
  `sobol_indices.csv`, `anova_conditional.csv`, `global_bands.csv`,
  `global_central.csv`, `robustness_sample_paths.csv`,
  `q25_q75_sample_paths.csv`
- Main figures: `figures/fig1-4*.pdf|png`
- Supplementary: `figures/per_country_dossier/*.pdf`
  (8 scenarios × 2 models, row-shared y-axis within each country/scenario),
  `figures/scenario_dashboards/*.pdf`

Downstream: `02_co2block_screening` reads `timeseries_central.csv`,
`q25_q75_sample_paths.csv`, and `robustness_sample_paths.csv`.
